# Multi-wavelength photonic-lantern propagation

Propagates a batch of DM-aberrated input fields through the lantern
at every wavelength of a native grid (one characterized `ChainPropagator`
per wavelength, built and released one at a time by cbeam's
`multi_wvl.MultiWavelengthPropagationPipeline`), giving each output fiber's
spectrum for each field.

The aberration here is a ramp of DM mode 0 (a petal piston) from -8 lambda to
+8 lambda: the physics behind Part B of `workbench.ipynb`. Its training set
(`<data>/spectral_41wl.npz`, random +-8 lambda pistons of all 5 petals) comes
from the same pipeline and the same characterizations, through
`ml_reconstruct.simulate.simulate_spectra` (chunked and parallel in
`gen_datasets.py` / `run_gen.sh`).

Imported from cbeam's `examples/multi_wavelength.ipynb` (branch
`mw-propagation`, 6a814ea), with its plotting helpers
(`multi_wavelength_vis.py`). `batch_pipeline` and `multi_wvl` come from the
cbeam checkout found by `ml_reconstruct.simulate` (`CBEAM_EXAMPLES`), and the
lantern characterizations are loaded from its `examples/data/`; each takes
minutes to compute where it is missing. The propagation results are cached
in `RESULTS_FILE`, so the notebook only redoes the plots unless
`RERUN = True`.

**Lantern.** The notebook runs for the lantern preset named by the environment
variable `CBEAM_LANTERN` (cbeam's `batch_pipeline.LANTERNS`): `pl19_800`, the
reference 19-port lantern at 800 nm (default); `pl7_1200`, 7 ports at
1200 nm; `pl19_1200`, 19 ports at 1200 nm in the same fibres as `pl7_1200`.
Its settings (piston range, polynomial budgets, data folder) are in
`ml_reconstruct.presets`; executed copies for each lantern are in
`runs/<lantern>/`.


In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import matplotlib.pyplot as plt

from ml_reconstruct.simulate import bp, ramp_spectra   # puts cbeam's examples/ on sys.path
import multi_wvl as mw
from multi_wavelength_vis import (
    plot_spectra_grid, plot_fiber_spectra, plot_wavelength_fiber_heatmap,
    plot_ramp_response, plot_total_throughput_vs_amplitude,
    plot_per_fiber_throughput_vs_amplitude, plot_core_vs_amplitude,
)

In [ ]:
from ml_reconstruct.presets import settings
S = settings(); LANTERN = S["lantern"]
base_params = bp.get_simulation_parameters(lantern=LANTERN)
WL_MIN_NM, WL_MAX_NM = base_params["band_nm"]
WVL_SAMPLES    = 41                    # native wavelengths (5 nm steps)
PISTON_SAMPLES = 81                    # steps of the mode-0 ramp
PISTON_MAX_NM  = S["spectral_span_nm"]  # ramp spans -PISTON_MAX_NM .. +PISTON_MAX_NM (8 lambda)

RESULTS_FILE = S["ramp"]
RERUN = False                          # True: propagate again even if RESULTS_FILE exists

## Optional: mode bookkeeping across the band

The lantern is characterized in two segments. The front one tracks every
mode, grouping the input-end degenerate pairs; the back one drops the mode
that has become a cladding mode by the junction and keeps one mode per
output core. `diagnose_mode_bookkeeping` prints the n_eff spectra at the
input and at the junction for a wavelength, without characterizing anything,
to check that picture before spending the time; it costs two FEM solves per
wavelength, so the scan is off by default.

In [ ]:
RUN_BOOKKEEPING_SCAN = False

native_wl_nm = np.linspace(WL_MIN_NM, WL_MAX_NM, WVL_SAMPLES)

if RUN_BOOKKEEPING_SCAN:
    lantern = mw.build_lantern_geometry(base_params)   # shared geometry, built once
    for wl in native_wl_nm:
        mw.diagnose_mode_bookkeeping(base_params, wl, PL_N=lantern)

## Propagate the ramp at every native wavelength

`get_power_spectra` returns, for every wavelength, field and output fiber,
the fraction of the incident power that reaches that fiber (the input
coupling efficiency is folded in, so the spectra are comparable across
wavelengths and fields).

In [ ]:
power_spectra, wl_out, coeff_matrix, labels = ramp_spectra(
    RESULTS_FILE, 0, PISTON_SAMPLES, PISTON_MAX_NM, native_wl_nm, rerun=RERUN, prop_chunk_size=32,
    lantern=LANTERN)
amplitudes = coeff_matrix[:, 0]
print(f"power_spectra: {power_spectra.shape} (wavelengths, fields, fibers)")

## Output spectra

For the unaberrated field (the middle of the ramp), then along the ramp.

In [ ]:
field_idx = PISTON_SAMPLES // 2          # zero piston

plot_spectra_grid(power_spectra, wl_out, field_idx=field_idx)
plot_fiber_spectra(power_spectra, wl_out, field_idx=field_idx,
                   fiber_indices=list(range(base_params["n_output_positions"])), plot_sum=True)
plot_wavelength_fiber_heatmap(power_spectra, wl_out, field_idx=field_idx)

In [ ]:
plot_ramp_response(power_spectra, wl_out, amplitudes, fiber_idx=0, labels=labels)
plot_total_throughput_vs_amplitude(power_spectra, amplitudes, labels=labels, mode_label="mode 0")
plot_per_fiber_throughput_vs_amplitude(power_spectra, amplitudes, mode_label="mode 0")

In [ ]:
# Central output core: one curve per wavelength along the ramp
plot_core_vs_amplitude(power_spectra, wl_out, amplitudes, core=0)

## Input coupling across the band

Summed over the fibers, the unaberrated field's spectrum is the fraction of
the incident power that couples into the lantern's tracked modes (propagation
is lossless apart from the light left in the cladding mode, which the back
segment drops), so it shows how the coupling changes with wavelength as the
PSF scales with lambda relative to the fixed input face.

In [ ]:
coupled = power_spectra[:, field_idx, :].sum(axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(wl_out, coupled, "o-")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Coupled fraction of incident power")
ax.set_title("Unaberrated field: input coupling vs. wavelength")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
for wl, c in list(zip(wl_out, coupled))[::5]:
    print(f"  {wl:5.0f} nm   {c:6.1%}")